In [2]:
import heapq
import math

CITIES = {
    "Delhi": (28.61, 77.21),
    "Jaipur": (26.91, 75.79),
    "Agra": (27.18, 78.01),
    "Lucknow": (26.85, 80.95),
    "Bhopal": (23.26, 77.41),
    "Ahmedabad": (23.02, 72.57),
    "Mumbai": (19.08, 72.88),
    "Pune": (18.52, 73.86),
    "Nagpur": (21.15, 79.09),
    "Hyderabad": (17.39, 78.49)
}

ROADS = [
    ("Delhi", "Jaipur", 280), ("Delhi", "Agra", 235), ("Delhi", "Lucknow", 555),
    ("Agra", "Jaipur", 240), ("Agra", "Lucknow", 330), ("Agra", "Bhopal", 570),
    ("Jaipur", "Ahmedabad", 660), ("Jaipur", "Bhopal", 600),
    ("Bhopal", "Ahmedabad", 580), ("Bhopal", "Nagpur", 350),
    ("Ahmedabad", "Mumbai", 530), ("Mumbai", "Pune", 150),
    ("Nagpur", "Hyderabad", 500), ("Pune", "Hyderabad", 560)
]

GRAPH = {city: [] for city in CITIES}
for a, b, km in ROADS:
    GRAPH[a].append((b, km))
    GRAPH[b].append((a, km))

def straight_line_km(city1, city2):
    lat1, lon1 = map(math.radians, CITIES[city1])
    lat2, lon2 = map(math.radians, CITIES[city2])
    a = (math.sin((lat2 - lat1) / 2) ** 2 +
         math.cos(lat1) * math.cos(lat2) *
         math.sin((lon2 - lon1) / 2) ** 2)
    return 6371 * 2 * math.asin(math.sqrt(a))

def check_heuristic():
    for a, b, km in ROADS:
        assert straight_line_km(a, b) <= km
    print("Heuristic check passed: air distance <= road distance on every road.")

def search(start, goal, use_astar):
    frontier = [(straight_line_km(start, goal), start, 0, [start])]
    best_g = {start: 0}
    expanded = 0

    while frontier:
        priority, city, g, path = heapq.heappop(frontier)

        if city == goal:
            return path, g, expanded

        if g > best_g.get(city, float("inf")):
            continue

        expanded += 1

        for neighbour, km in GRAPH[city]:
            new_g = g + km

            if new_g < best_g.get(neighbour, float("inf")):
                best_g[neighbour] = new_g
                h = straight_line_km(neighbour, goal)
                priority = new_g + h if use_astar else h
                heapq.heappush(frontier, (priority, neighbour, new_g, path + [neighbour]))

    return None, float("inf"), expanded

if __name__ == "__main__":
    check_heuristic()

    print("\n" + "=" * 70)
    print("WORKED EXAMPLE: Lucknow to Pune")
    print("=" * 70)

    for name, flag in [("Greedy Best-First", False), ("A*", True)]:
        path, km, nodes = search("Lucknow", "Pune", flag)
        print(f"{name:<18}: {' -> '.join(path)}")
        print(f"{'':<18}  distance = {km} km, cities expanded = {nodes}\n")

    print("=" * 70)
    print("ANALYSIS: Greedy Best-First vs A*")
    print("=" * 70)
    print(f"{'Trip':<22} | {'Greedy km (exp)':>16} | {'A* km (exp)':>14} | Greedy shortest?")
    print("-" * 70)

    trips = [
        ("Delhi", "Pune"),
        ("Lucknow", "Pune"),
        ("Agra", "Hyderabad"),
        ("Jaipur", "Nagpur"),
        ("Lucknow", "Mumbai"),
        ("Delhi", "Hyderabad")
    ]

    for s, g in trips:
        _, gk, gn = search(s, g, False)
        _, ak, an = search(s, g, True)
        print(f"{s + ' -> ' + g:<22} | {f'{gk} ({gn})':>16} | {f'{ak} ({an})':>14} | {'Yes' if gk == ak else 'NO'}")

    print("\nObservations:")
    print(" - Greedy Best-First uses only h(n), the estimated distance to the goal.")
    print(" - A* uses g(n) + h(n), considering both travelled and estimated distance.")
    print(" - A* finds the shortest route when the heuristic is admissible.")
    print(" - Greedy Best-First can be faster but may produce a longer route.")

Heuristic check passed: air distance <= road distance on every road.

WORKED EXAMPLE: Lucknow to Pune
Greedy Best-First : Lucknow -> Agra -> Bhopal -> Ahmedabad -> Mumbai -> Pune
                    distance = 2160 km, cities expanded = 5

A*                : Lucknow -> Agra -> Jaipur -> Ahmedabad -> Mumbai -> Pune
                    distance = 1910 km, cities expanded = 8

ANALYSIS: Greedy Best-First vs A*
Trip                   |  Greedy km (exp) |    A* km (exp) | Greedy shortest?
----------------------------------------------------------------------
Delhi -> Pune          |         1620 (4) |       1620 (6) | Yes
Lucknow -> Pune        |         2160 (5) |       1910 (8) | NO
Agra -> Hyderabad      |         1420 (3) |       1420 (5) | Yes
Jaipur -> Nagpur       |          950 (2) |        950 (3) | Yes
Lucknow -> Mumbai      |         2010 (4) |       1760 (6) | NO
Delhi -> Hyderabad     |         1655 (5) |       1655 (6) | Yes

Observations:
 - Greedy Best-First uses only h(n),